## Step 1 — screen blocks for the block-splitting workflow
**# of cells in notebook:** 7

**Purpose:** Identify blocks that should enter the block-splitting workflow. Buildings are first analyzed using Optimized Hot Spot Analysis based on building area. The building-level hot-spot/cold-spot results are then summarized by block and combined with block area and population criteria to identify blocks for further analysis.

**Input:**

- a buildings layer with building area
- a citywide blocks layer with a unique `block_id` and `population`
- a geodatabase for the Optimized Hot Spot Analysis output

**Output:**

- `high_high_low_low_optimized` — buildings with Optimized Hot Spot Analysis results, including `Gi_Bin`
- `{city}_blocks_utm{zone}_GiBin` — blocks with counts of buildings in each `Gi_Bin` class
- `{city}_blocks_utm{zone}_GiBin_flags` — blocks with screening flag fields
- `heterogeneous_largePop_blocks` — blocks selected for the block-splitting workflow

**Main logic:**

**Cell 1 — Define inputs, outputs, fields, and screening thresholds**

1. Defines the buildings, blocks, and output datasets.
2. Defines the building-area field, block ID field, population field, Gi_Bin count fields, and screening flags.
3. Sets the block-area threshold to 100,000 m² (10 hectares) and the population threshold to 1,000.

**Cell 2 — Define helper functions**

1. Defines functions used to check datasets and fields, delete existing outputs, count features, and print dataset information.

**Cell 3 — Run Optimized Hot Spot Analysis**

1. Runs ArcGIS Optimized Hot Spot Analysis on building footprints using building area as the analysis field.
2. Confirms that the output contains `Gi_Bin`.
3. Reports the number of buildings in each `Gi_Bin` category.

**Cell 4 — Summarize building Gi_Bin classes by block**

1. Converts analyzed building polygons to inside points.
2. Spatially joins the building points to blocks.
3. Counts buildings in each `Gi_Bin` class (-3 through +3) for every block.
4. Adds the seven Gi_Bin count fields to a copy of the citywide blocks layer and performs a count audit.

**Cell 5 — Create block screening flags**

1. Uses `Gi_Minus3` and `Gi_Plus3` to identify blocks containing 99% confidence cold spots and hot spots.
2. Creates the mutually exclusive heterogeneity flags `HH_CC`, `HH_Grtr10ha`, and `CC_Grtr10ha`.
3. Creates `LargePop` for blocks with population greater than 1,000.

**Cell 6 — Create the block-splitting selection**

1. Selects blocks where at least one screening flag equals 1.
2. Writes the selected features to `heterogeneous_largePop_blocks`.

**Cell 7 — Final QA**

1. Confirms that the expected output datasets exist and reports their feature counts.
2. Confirms that every selected block has at least one active screening flag.
3. Checks that the three heterogeneity flags remain mutually exclusive.


In [ ]:
import os
from collections import Counter, defaultdict

import arcpy

arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False

# ---------------------------------------------------------------------
# INPUTS / OUTPUTS
# ---------------------------------------------------------------------

BUILDINGS = (
    r"E:\_kigali\_analysis\population\population.gdb"
    r"\kigali_buildings_utm36s"
)

HOTSPOT_GDB = (
    r"E:\_kigali\_analysis\gi_star_buildings_full"
    r"\high-high low-low getis.gdb"
)

HOTSPOT_OUTPUT = os.path.join(
    HOTSPOT_GDB,
    "high_high_low_low_optimized"
)

BLOCKS_GDB = r"E:\_kigali\_analysis\blocks\blocks.gdb"

BLOCKS = os.path.join(
    BLOCKS_GDB,
    "kigali_blocks_utm36s"
)

BLOCKS_GIBIN = os.path.join(
    BLOCKS_GDB,
    "kigali_blocks_utm36s_GiBin"
)

BLOCKS_FLAGS = os.path.join(
    BLOCKS_GDB,
    "kigali_blocks_utm36s_GiBin_flags"
)

FINAL_SELECTION = os.path.join(
    BLOCKS_GDB,
    "heterogeneous_largePop_blocks"
)

# ---------------------------------------------------------------------
# FIELD / THRESHOLD SETTINGS
# ---------------------------------------------------------------------

ANALYSIS_FIELD = "area_m_utm"
BLOCK_ID_FIELD = "block_id"
POPULATION_FIELD = "population"

AREA_THRESHOLD_M2 = 100_000.0       # 10 hectares
POPULATION_THRESHOLD = 1_000.0      # LargePop means population > 1000

GI_COUNT_FIELDS = {
    -3: "Gi_Minus3",
    -2: "Gi_Minus2",
    -1: "Gi_Minus1",
     0: "Gi_0",
     1: "Gi_Plus1",
     2: "Gi_Plus2",
     3: "Gi_Plus3",
}

FLAG_FIELDS = [
    "HH_CC",
    "HH_Grtr10ha",
    "CC_Grtr10ha",
    "LargePop",
]

print("Settings loaded.")
print(f"Buildings:        {BUILDINGS}")
print(f"Hotspot output:   {HOTSPOT_OUTPUT}")
print(f"Blocks:           {BLOCKS}")
print(f"GiBin blocks:     {BLOCKS_GIBIN}")
print(f"Flagged blocks:   {BLOCKS_FLAGS}")
print(f"Final selection:  {FINAL_SELECTION}")

In [ ]:
# ---------------------------------------------------------------------
# HELPER FUNCTIONS
# ---------------------------------------------------------------------

def delete_if_exists(path):
    if arcpy.Exists(path):
        arcpy.management.Delete(path)


def field_names(dataset):
    return [f.name for f in arcpy.ListFields(dataset)]


def get_field_name(dataset, desired_name):
    lookup = {f.name.lower(): f.name for f in arcpy.ListFields(dataset)}
    return lookup.get(desired_name.lower())


def require_fields(dataset, required_fields):
    missing = [
        f for f in required_fields
        if get_field_name(dataset, f) is None
    ]
    if missing:
        raise ValueError(
            f"{dataset}\n"
            f"is missing required field(s): {missing}"
        )


def require_exists(path, label):
    if not arcpy.Exists(path):
        raise FileNotFoundError(f"{label} does not exist:\n{path}")


def get_count(dataset):
    return int(arcpy.management.GetCount(dataset)[0])


def describe_dataset(path, label):
    desc = arcpy.Describe(path)
    sr = desc.spatialReference
    print(f"\n{label}")
    print(f"  Path:          {path}")
    print(f"  Features:      {get_count(path):,}")
    print(f"  Shape type:    {desc.shapeType}")
    print(f"  CRS:           {sr.name}")
    print(f"  Linear units:  {getattr(sr, 'linearUnitName', 'unknown')}")


# Validate the major inputs now.
require_exists(BUILDINGS, "Buildings input")
require_exists(HOTSPOT_GDB, "Hotspot output geodatabase")
require_exists(BLOCKS, "Blocks input")

require_fields(BUILDINGS, [ANALYSIS_FIELD])
require_fields(BLOCKS, [BLOCK_ID_FIELD, POPULATION_FIELD])

describe_dataset(BUILDINGS, "Buildings input")
describe_dataset(BLOCKS, "Blocks input")

print("\nInput validation complete.")

In [ ]:
# =====================================================================
# STEP 1 — OPTIMIZED HOT SPOT ANALYSIS ON BUILDINGS
# =====================================================================

print("=" * 80)
print("STEP 1: OPTIMIZED HOT SPOT ANALYSIS")
print("=" * 80)

delete_if_exists(HOTSPOT_OUTPUT)

result = arcpy.stats.OptimizedHotSpotAnalysis(
    Input_Features=BUILDINGS,
    Output_Features=HOTSPOT_OUTPUT,
    Analysis_Field=ANALYSIS_FIELD,
)

print(result.getMessages())
print(f"\nCreated:\n  {HOTSPOT_OUTPUT}")

# Validate expected output field.
gi_bin_field = get_field_name(HOTSPOT_OUTPUT, "Gi_Bin")
if gi_bin_field is None:
    raise RuntimeError(
        "Optimized Hot Spot Analysis completed, but no Gi_Bin field "
        "was found in the output."
    )

# Print the Gi_Bin distribution.
gi_distribution = Counter()

with arcpy.da.SearchCursor(HOTSPOT_OUTPUT, [gi_bin_field]) as cursor:
    for (value,) in cursor:
        if value is None:
            gi_distribution[None] += 1
        else:
            gi_distribution[int(value)] += 1

print("\nGi_Bin distribution:")
for gi_value in [-3, -2, -1, 0, 1, 2, 3]:
    print(f"  {gi_value:>2}: {gi_distribution.get(gi_value, 0):,}")

if gi_distribution.get(None, 0):
    print(f"  NULL: {gi_distribution[None]:,}")

describe_dataset(HOTSPOT_OUTPUT, "Optimized Hot Spot Analysis output")

In [ ]:
# =====================================================================
# STEP 2 — COUNT BUILDING Gi_Bin CATEGORIES WITHIN EACH BLOCK
#          Output: kigali_blocks_utm36s_GiBin
# =====================================================================

print("=" * 80)
print("STEP 2: AGGREGATE BUILDING Gi_Bin COUNTS TO BLOCKS")
print("=" * 80)

require_exists(HOTSPOT_OUTPUT, "Optimized Hot Spot Analysis output")
require_fields(HOTSPOT_OUTPUT, ["Gi_Bin"])
require_fields(BLOCKS, [BLOCK_ID_FIELD])

# Ensure block_id is populated and unique.
block_id_actual = get_field_name(BLOCKS, BLOCK_ID_FIELD)

block_ids = []
null_block_ids = 0

with arcpy.da.SearchCursor(BLOCKS, [block_id_actual]) as cursor:
    for (block_id,) in cursor:
        if block_id is None or str(block_id).strip() == "":
            null_block_ids += 1
        else:
            block_ids.append(str(block_id))

if null_block_ids:
    raise ValueError(
        f"{null_block_ids:,} block(s) have null/blank {BLOCK_ID_FIELD} values."
    )

if len(block_ids) != len(set(block_ids)):
    raise ValueError(f"{BLOCK_ID_FIELD} is not unique in the input blocks.")

# Temporary feature classes in the ArcGIS scratch geodatabase.
scratch_gdb = arcpy.env.scratchGDB

blocks_for_join = arcpy.CreateUniqueName(
    "tmp_kigali_blocks_gibin_join",
    scratch_gdb
)

building_points = arcpy.CreateUniqueName(
    "tmp_kigali_hotspot_bldg_points",
    scratch_gdb
)

points_to_blocks_sj = arcpy.CreateUniqueName(
    "tmp_kigali_hotspot_points_blocks_sj",
    scratch_gdb
)

TEMP_BLOCK_ID_FIELD = "src_blk_id"

try:
    # -------------------------------------------------------------
    # 2A. Make a temporary block copy carrying an unmistakable
    #     block identifier for the spatial join.
    # -------------------------------------------------------------
    print("\nCreating temporary blocks for the spatial join...")

    delete_if_exists(blocks_for_join)
    arcpy.management.CopyFeatures(BLOCKS, blocks_for_join)

    if get_field_name(blocks_for_join, TEMP_BLOCK_ID_FIELD):
        arcpy.management.DeleteField(
            blocks_for_join,
            TEMP_BLOCK_ID_FIELD
        )

    arcpy.management.AddField(
        blocks_for_join,
        TEMP_BLOCK_ID_FIELD,
        "TEXT",
        field_length=255
    )

    copied_block_id_field = get_field_name(
        blocks_for_join,
        BLOCK_ID_FIELD
    )

    arcpy.management.CalculateField(
        blocks_for_join,
        TEMP_BLOCK_ID_FIELD,
        f"str(!{copied_block_id_field}!)",
        "PYTHON3"
    )

    # -------------------------------------------------------------
    # 2B. Convert every hotspot building polygon to an inside point.
    #     FeatureToPoint preserves Gi_Bin.
    # -------------------------------------------------------------
    print("Creating inside points for hotspot buildings...")

    delete_if_exists(building_points)

    arcpy.management.FeatureToPoint(
        in_features=HOTSPOT_OUTPUT,
        out_feature_class=building_points,
        point_location="INSIDE"
    )

    print(
        f"  Building polygons: {get_count(HOTSPOT_OUTPUT):,}\n"
        f"  Building points:   {get_count(building_points):,}"
    )

    # -------------------------------------------------------------
    # 2C. Spatially join each building point to a block.
    #     JOIN_ONE_TO_ONE guarantees one output record per building.
    # -------------------------------------------------------------
    print("Spatially joining building points to blocks...")

    delete_if_exists(points_to_blocks_sj)

    arcpy.analysis.SpatialJoin(
        target_features=building_points,
        join_features=blocks_for_join,
        out_feature_class=points_to_blocks_sj,
        join_operation="JOIN_ONE_TO_ONE",
        join_type="KEEP_ALL",
        match_option="INTERSECT"
    )

    sj_gi_field = get_field_name(points_to_blocks_sj, "Gi_Bin")
    sj_block_field = get_field_name(
        points_to_blocks_sj,
        TEMP_BLOCK_ID_FIELD
    )
    sj_join_count_field = get_field_name(
        points_to_blocks_sj,
        "Join_Count"
    )

    if sj_gi_field is None or sj_block_field is None:
        raise RuntimeError(
            "Spatial join output is missing Gi_Bin and/or "
            f"{TEMP_BLOCK_ID_FIELD}."
        )

    # -------------------------------------------------------------
    # 2D. Build a block_id -> Gi_Bin count dictionary.
    # -------------------------------------------------------------
    counts_by_block = defaultdict(Counter)
    unmatched_buildings = 0
    multi_match_buildings = 0
    unexpected_bins = Counter()

    cursor_fields = [
        sj_block_field,
        sj_gi_field,
    ]

    if sj_join_count_field:
        cursor_fields.append(sj_join_count_field)

    with arcpy.da.SearchCursor(
        points_to_blocks_sj,
        cursor_fields
    ) as cursor:

        for row in cursor:
            block_id = row[0]
            gi_value = row[1]

            join_count = (
                row[2]
                if len(cursor_fields) == 3
                else None
            )

            if join_count is not None and int(join_count) > 1:
                multi_match_buildings += 1

            if block_id is None or str(block_id).strip() == "":
                unmatched_buildings += 1
                continue

            if gi_value is None:
                unexpected_bins[None] += 1
                continue

            gi_int = int(gi_value)

            if gi_int not in GI_COUNT_FIELDS:
                unexpected_bins[gi_int] += 1
                continue

            counts_by_block[str(block_id)][gi_int] += 1

    print(f"\nUnmatched building points: {unmatched_buildings:,}")

    if multi_match_buildings:
        print(
            "WARNING: "
            f"{multi_match_buildings:,} building point(s) intersected "
            "more than one block. JOIN_ONE_TO_ONE retained one joined record."
        )

    if unexpected_bins:
        print(f"Unexpected/NULL Gi_Bin values: {dict(unexpected_bins)}")

    # -------------------------------------------------------------
    # 2E. Copy the original blocks and add the seven count fields.
    # -------------------------------------------------------------
    print(f"\nCreating output blocks:\n  {BLOCKS_GIBIN}")

    delete_if_exists(BLOCKS_GIBIN)
    arcpy.management.CopyFeatures(BLOCKS, BLOCKS_GIBIN)

    for output_field in GI_COUNT_FIELDS.values():
        if get_field_name(BLOCKS_GIBIN, output_field) is None:
            arcpy.management.AddField(
                BLOCKS_GIBIN,
                output_field,
                "LONG"
            )

    out_block_id = get_field_name(
        BLOCKS_GIBIN,
        BLOCK_ID_FIELD
    )

    out_count_fields = [
        get_field_name(BLOCKS_GIBIN, name)
        for name in GI_COUNT_FIELDS.values()
    ]

    cursor_fields = [out_block_id] + out_count_fields

    updated_blocks = 0

    with arcpy.da.UpdateCursor(
        BLOCKS_GIBIN,
        cursor_fields
    ) as cursor:

        for row in cursor:
            block_id = str(row[0])
            c = counts_by_block.get(block_id, Counter())

            row[1:] = [
                int(c.get(gi_value, 0))
                for gi_value in GI_COUNT_FIELDS.keys()
            ]

            cursor.updateRow(row)
            updated_blocks += 1

    print(f"Blocks updated: {updated_blocks:,}")

    # -------------------------------------------------------------
    # 2F. Basic count audit.
    # -------------------------------------------------------------
    total_counted_in_blocks = 0

    with arcpy.da.SearchCursor(
        BLOCKS_GIBIN,
        out_count_fields
    ) as cursor:
        for row in cursor:
            total_counted_in_blocks += sum(
                0 if v is None else int(v)
                for v in row
            )

    expected_matched = get_count(building_points) - unmatched_buildings

    print("\nGi_Bin aggregation audit:")
    print(f"  Building points created:        {get_count(building_points):,}")
    print(f"  Building points unmatched:      {unmatched_buildings:,}")
    print(f"  Expected assigned buildings:    {expected_matched:,}")
    print(f"  Sum of seven block Gi counts:   {total_counted_in_blocks:,}")

    if total_counted_in_blocks != expected_matched:
        print(
            "WARNING: total block Gi counts differ from the expected "
            "number of assigned building points."
        )
    else:
        print("  Audit result: PASS")

finally:
    # Clean temporary scratch outputs.
    for temp_fc in [
        blocks_for_join,
        building_points,
        points_to_blocks_sj,
    ]:
        delete_if_exists(temp_fc)

print(f"\nCreated:\n  {BLOCKS_GIBIN}")

In [ ]:
# =====================================================================
# STEP 3 — CREATE HH / CC / AREA / POPULATION FLAGS
#          Output: kigali_blocks_utm36s_GiBin_flags
# =====================================================================

print("=" * 80)
print("STEP 3: CREATE SCREENING FLAGS")
print("=" * 80)

require_exists(BLOCKS_GIBIN, "Gi_Bin block layer")
require_fields(
    BLOCKS_GIBIN,
    list(GI_COUNT_FIELDS.values()) + [POPULATION_FIELD]
)

# Because the area rule is 10 hectares = 100,000 m², confirm projected
# linear units are meters.
sr = arcpy.Describe(BLOCKS_GIBIN).spatialReference
linear_unit = getattr(sr, "linearUnitName", "")

print(f"CRS: {sr.name}")
print(f"Linear units: {linear_unit}")

if "meter" not in str(linear_unit).lower():
    raise ValueError(
        "The block layer does not appear to use meter-based projected "
        "coordinates. The 100,000 m² threshold should not be applied."
    )

delete_if_exists(BLOCKS_FLAGS)
arcpy.management.CopyFeatures(
    BLOCKS_GIBIN,
    BLOCKS_FLAGS
)

for flag_field in FLAG_FIELDS:
    if get_field_name(BLOCKS_FLAGS, flag_field) is None:
        arcpy.management.AddField(
            BLOCKS_FLAGS,
            flag_field,
            "SHORT"
        )

gi_minus3 = get_field_name(BLOCKS_FLAGS, "Gi_Minus3")
gi_plus3 = get_field_name(BLOCKS_FLAGS, "Gi_Plus3")
population_field = get_field_name(
    BLOCKS_FLAGS,
    POPULATION_FIELD
)

hh_cc_field = get_field_name(BLOCKS_FLAGS, "HH_CC")
hh_gt10_field = get_field_name(
    BLOCKS_FLAGS,
    "HH_Grtr10ha"
)
cc_gt10_field = get_field_name(
    BLOCKS_FLAGS,
    "CC_Grtr10ha"
)
largepop_field = get_field_name(
    BLOCKS_FLAGS,
    "LargePop"
)

cursor_fields = [
    gi_minus3,
    gi_plus3,
    population_field,
    "SHAPE@AREA",
    hh_cc_field,
    hh_gt10_field,
    cc_gt10_field,
    largepop_field,
]

flag_counts = Counter()

with arcpy.da.UpdateCursor(
    BLOCKS_FLAGS,
    cursor_fields
) as cursor:

    for row in cursor:
        minus3 = 0 if row[0] is None else int(row[0])
        plus3 = 0 if row[1] is None else int(row[1])
        population = 0.0 if row[2] is None else float(row[2])
        area_m2 = float(row[3])

        # 99% confidence cold/hot presence.
        has_cc_99 = minus3 > 0
        has_hh_99 = plus3 > 0

        # Reconstructed mutually exclusive heterogeneity flags:
        #
        # HH_CC:
        #   at least one 99% hot-spot building AND
        #   at least one 99% cold-spot building.
        #
        # HH_Grtr10ha:
        #   99% hot spot present, no 99% cold spot present,
        #   and block area > 10 ha.
        #
        # CC_Grtr10ha:
        #   99% cold spot present, no 99% hot spot present,
        #   and block area > 10 ha.
        #
        # LargePop can overlap any of the above.
        hh_cc = int(has_hh_99 and has_cc_99)

        hh_grtr10ha = int(
            has_hh_99
            and not has_cc_99
            and area_m2 > AREA_THRESHOLD_M2
        )

        cc_grtr10ha = int(
            has_cc_99
            and not has_hh_99
            and area_m2 > AREA_THRESHOLD_M2
        )

        large_pop = int(
            population > POPULATION_THRESHOLD
        )

        row[4] = hh_cc
        row[5] = hh_grtr10ha
        row[6] = cc_grtr10ha
        row[7] = large_pop

        cursor.updateRow(row)

        flag_counts["HH_CC"] += hh_cc
        flag_counts["HH_Grtr10ha"] += hh_grtr10ha
        flag_counts["CC_Grtr10ha"] += cc_grtr10ha
        flag_counts["LargePop"] += large_pop

print(f"\nCreated:\n  {BLOCKS_FLAGS}")

print("\nFlag counts:")
for flag in FLAG_FIELDS:
    print(f"  {flag:14s}: {flag_counts[flag]:,}")

In [ ]:
# =====================================================================
# STEP 4 — SELECT ALL BLOCKS MEETING AT LEAST ONE FLAG
#          Output: heterogeneous_largePop_blocks
# =====================================================================

print("=" * 80)
print("STEP 4: CREATE FINAL HETEROGENEOUS / LARGE-POPULATION SELECTION")
print("=" * 80)

require_exists(BLOCKS_FLAGS, "Flagged block layer")
require_fields(BLOCKS_FLAGS, FLAG_FIELDS)

delete_if_exists(FINAL_SELECTION)

flags_layer = "kigali_gibin_flags_lyr"

if arcpy.Exists(flags_layer):
    arcpy.management.Delete(flags_layer)

arcpy.management.MakeFeatureLayer(
    BLOCKS_FLAGS,
    flags_layer
)

where_clause = " OR ".join(
    f"{arcpy.AddFieldDelimiters(BLOCKS_FLAGS, flag)} = 1"
    for flag in FLAG_FIELDS
)

print(f"Selection expression:\n  {where_clause}")

arcpy.management.SelectLayerByAttribute(
    flags_layer,
    "NEW_SELECTION",
    where_clause
)

selected_count = get_count(flags_layer)
source_count = get_count(BLOCKS_FLAGS)

print(f"\nSource blocks:   {source_count:,}")
print(f"Selected blocks: {selected_count:,}")

arcpy.management.CopyFeatures(
    flags_layer,
    FINAL_SELECTION
)

arcpy.management.Delete(flags_layer)

print(f"\nCreated:\n  {FINAL_SELECTION}")
print(f"Final feature count: {get_count(FINAL_SELECTION):,}")

In [ ]:
# =====================================================================
# STEP 5 — FINAL QA
# =====================================================================

print("=" * 80)
print("STEP 5: FINAL QA")
print("=" * 80)

for dataset, label in [
    (HOTSPOT_OUTPUT, "Optimized hotspot buildings"),
    (BLOCKS_GIBIN, "Blocks with Gi_Bin counts"),
    (BLOCKS_FLAGS, "Blocks with flags"),
    (FINAL_SELECTION, "Final selected blocks"),
]:
    require_exists(dataset, label)
    print(f"{label:28s}: {get_count(dataset):,}")

# Confirm every feature in FINAL_SELECTION satisfies at least one flag.
bad_final_rows = 0
multiple_heterogeneity_flags = 0

final_flag_fields = [
    get_field_name(FINAL_SELECTION, flag)
    for flag in FLAG_FIELDS
]

with arcpy.da.SearchCursor(
    FINAL_SELECTION,
    final_flag_fields
) as cursor:

    for row in cursor:
        vals = [
            0 if v is None else int(v)
            for v in row
        ]

        if sum(vals) == 0:
            bad_final_rows += 1

        # First three are the mutually exclusive heterogeneity flags.
        if sum(vals[:3]) > 1:
            multiple_heterogeneity_flags += 1

print(f"\nFinal rows with no active flag: {bad_final_rows:,}")
print(
    "Final rows with >1 heterogeneity flag: "
    f"{multiple_heterogeneity_flags:,}"
)

if bad_final_rows == 0 and multiple_heterogeneity_flags == 0:
    print("\nFINAL QA: PASS")
else:
    print("\nFINAL QA: REVIEW WARNINGS ABOVE")